# Transfer learning met PyTorch

In deze notebook gaan we het concept van transfer learning onderzoeken. Transfer learning is een techniek waarbij een pre-trained neuraal netwerk wordt gebruikt als startpunt voor een nieuwe taak. Er zijn twee belangrijke benaderingen binnen transfer learning:

1. **Feature Extractie**: Hierbij houden we de gewichten van het pre-trained model vast en gebruiken we het model als feature extractor. Alleen het laatste (geclassificeerde) deel van het netwerk wordt aangepast aan de nieuwe taak.
  
2. **Fine-Tuning**: Hierbij worden de gewichten van (een deel van) het pre-trained model verder getraind op de nieuwe taak. Dit maakt het mogelijk om het model verder aan te passen aan de specifieke data.

## Stappen in de notebook

1. **Data voorbereiding**:
   - We gebruiken een kleine dataset voor classificatie (bijvoorbeeld een subset van de CIFAR-10 dataset).

2. **Model laden en voorbereiden**:
   - We laden een pre-trained model (bijv. ResNet-18) uit PyTorch's `torchvision` bibliotheek.
   - We maken aanpassingen voor beide benaderingen van transfer learning.

3. **Training**:
   - We demonstreren zowel de **feature extractie** als de **fine-tuning** methode.
   - We trainen de laatste laag voor de feature extractie en een deel van het netwerk voor fine-tuning.

4. **Evaluatie**:
   - We evalueren de prestaties van beide benaderingen op een testset.

---

## 1. Data Voorbereiding

We beginnen met het importeren van de benodigde bibliotheken en het laden van de dataset.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import random

# Data transformaties
transform = transforms.Compose([
    transforms.Resize(224),  # Resize to 224x224 pixels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# of: ResNet18_Weights.IMAGENET1K_V1.transforms (doet de transforms ook)

# Laden van de CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
total_samples = len(trainset)
sample_size = int(0.01 * total_samples)
random_indices = random.sample(range(total_samples), sample_size)
trainset_reduced = Subset(trainset, random_indices)
trainloader = DataLoader(trainset_reduced, batch_size=32, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
total_samples = len(testset)
sample_size = int(0.01 * total_samples)
random_indices = random.sample(range(total_samples), sample_size)
testset_reduced = Subset(testset, random_indices)

# Create a DataLoader for the reduced dataset
testloader = DataLoader(testset_reduced, batch_size=32, shuffle=True, num_workers=2)

# Klassen in CIFAR-10
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [6]:

from torchinfo import summary

summary(model_ft, input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 10]                   --
├─Conv2d: 1-1                            [1, 64, 112, 112]         (9,408)
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         (128)
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 64, 56, 56]           --
│    └─BasicBlock: 2-1                   [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           (36,864)
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           (128)
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           (36,864)
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           (128)
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]   

## 2. Model Laden en Voorbereiden

### A) Feature Extractie

In deze aanpak houden we de gewichten van het pre-trained model vast en passen we alleen het fully-connected gedeelte aan.

In [5]:
import torchvision.models as models

# Laden van het pre-trained ResNet-18 model
model_ft = models.resnet18(pretrained=True)

#print(model_ft)

# Vastzetten van alle lagen, behalve de laatste
for param in model_ft.parameters():
    param.requires_grad = False

# Vervangen van de laatste volledig verbonden laag
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(classes))  # 10 klassen in CIFAR-10 - past het fully connected gedeelte aan

print(model_ft)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

### B) Fine-Tuning

In deze aanpak staan we toe dat sommige (of alle) gewichten verder worden getraind.

In [ ]:
# Laden van het pre-trained ResNet-18 model voor fine-tuning
model_ft_finetune = models.resnet18(pretrained=True)

# Alleen de laatste paar lagen ontgrendelen voor training
# Bijvoorbeeld, alle lagen vanaf layer4 worden getraind
for name, param in model_ft_finetune.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Vervangen van de laatste volledig verbonden laag
model_ft_finetune.fc = nn.Linear(num_ftrs, len(classes))

## 3. Training

Hier zullen we beide modellen trainen, eerst het model voor feature extractie, gevolgd door het model voor fine-tuning.

### A) Feature Extractie Training

In [ ]:
import time

# Definieer een loss-functie en een optimizer
criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.Adam(model_ft.fc.parameters(), lr=0.001)

# Trainingslus voor feature extractie
num_epochs = 5
model_ft.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    
    start_time = time.time()  # Start tijd opnemen
    
    for inputs, labels in trainloader:
        optimizer_ft.zero_grad()

        # Voorwaartse pass
        outputs = model_ft(inputs)
        loss = criterion(outputs, labels)

        # Achterwaartse pass + optimalisatie
        loss.backward()
        optimizer_ft.step()

        running_loss += loss.item()
            
    # Eind tijd opnemen
    end_time = time.time()
    epoch_duration = end_time - start_time  # Tijd per epoch berekenen
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}, Duration: {epoch_duration:.2f} seconds")

print("Feature Extractie Model is getraind.")


### B) Fine-Tuning Training

In [ ]:
# Definieer een loss-functie en een optimizer voor fine-tuning
optimizer_ft_finetune = optim.SGD(model_ft_finetune.parameters(), lr=0.001, momentum=0.9)

# Trainingslus voor fine-tuning
model_ft_finetune.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    start_time = time.time()  # Start tijd opnemen
    
    for inputs, labels in trainloader:
        optimizer_ft_finetune.zero_grad()

        # Voorwaartse pass
        outputs = model_ft_finetune(inputs)
        loss = criterion(outputs, labels)

        # Achterwaartse pass + optimalisatie
        loss.backward()
        optimizer_ft_finetune.step()

        running_loss += loss.item()
            
    # Eind tijd opnemen
    end_time = time.time()
    epoch_duration = end_time - start_time  # Tijd per epoch berekenen
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}, Duration: {epoch_duration:.2f} seconds")

print("Fine-Tuning Model is getraind.")


## 4. Evaluatie
Na het trainen evalueren we de prestaties van beide modellen op de testset.

### A) Evaluatie van het Feature Extractie Model

In [ ]:
correct = 0
total = 0
model_ft.eval()

with torch.no_grad():
    for inputs, labels in testloader:
        outputs = model_ft(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Feature Extractie Model Accuraatheid: {100 * correct / total:.2f}%')

### B) Evaluatie van het Fine-Tuning Model

In [ ]:
correct = 0
total = 0
model_ft_finetune.eval()

with torch.no_grad():
    for inputs, labels in testloader:
        outputs = model_ft_finetune(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Fine-Tuning Model Accuraatheid: {100 * correct / total:.2f}%')


## Oefening: transfer learning met extra lagen voor FashionMNIST

### Oefeningomschrijving:
In deze oefening ga je een neuraal netwerk trainen met de FashionMNIST dataset, gebruikmakend van transfer learning. Je zult een pre-trained ResNet-18 model gebruiken, en daar drie extra volledig verbonden lagen aan toevoegen. Vervolgens train je het model en evalueer je de prestaties.

### Stappen:

**Data Voorbereiding:**

Laad de FashionMNIST dataset en breng de nodige transformaties aan.
Splits de data in een trainingsset en een testset.

**Model Voorbereiding:**

Laad een pre-trained ResNet-18 model.
Pas het model aan door drie extra volledig verbonden lagen toe te voegen:
* De eerste extra laag moet 512 neuronen hebben en een ReLU activatiefunctie.
* De tweede extra laag moet 256 neuronen hebben en een ReLU activatiefunctie.
* De derde extra laag moet 128 neuronen hebben en een ReLU activatiefunctie.
* De laatste output laag moet het aantal klassen in FashionMNIST (10 klassen) bevatten.

**Training:**

Definieer een loss-functie en een optimizer.
Train het model voor een aantal epochs.
Meet en print de tijd die elke epoch kost.

**Evaluatie:**

Evalueer de prestaties van het getrainde model op de testset en rapporteer de accuraatheid.

**Vragen om te beantwoorden:**

* Wat is het effect van de extra lagen op de prestaties van het model?
* Hoeveel tijd kost elke epoch en hoe varieert dit met het aantal lagen?